In [1]:
import secrets
import os
import pathlib
import base64

In [3]:
def gerar_chave(tamanho: int) -> bytes:
    """Gera uma chave criptograficamente segura com tamanho bytes."""
    chave_gerada_aleatoriamente = secrets.token_bytes(tamanho)
    return chave_gerada_aleatoriamente

def xor_bytes(dados: bytes, chave: bytes) -> bytes:
    """Calcula o XOR entre sequencias de mesmo comprimento."""
    if(len(dados)==len(chave)):
        # resultado_xor = bytearray()
        resultado_xor = []
        for i in range(len(dados)):
            resultado_xor.append(dados[i]^chave[i]) 
        return bytes(resultado_xor) #melhor deixar a conversão de bytes aqui já e resolver mais bunitinho do que antes
    else:
        raise ValueError("Erro! Tamanhos diferentes entre dados e chave!")


def cifrar(mensagem: bytes, chave: bytes) -> bytes:
    """Cifra mensagem usando OTP."""
    if(len(mensagem)==len(chave)):
        cifrado_calculado = xor_bytes(mensagem,chave)
        return cifrado_calculado
    else:
        raise ValueError("Erro! Tamanhos diferentes entre mensagem e chave!")

def decifrar(cifrado: bytes, chave: bytes) -> bytes:
    """Decifra um texto cifrado usando OTP."""
    if(len(cifrado)==len(chave)):
        mensagem_decifrada = xor_bytes(cifrado,chave)
        return mensagem_decifrada
    else:
        raise ValueError("Erro! Tamanhos diferentes entre cifrado e chave!")

In [4]:
import secrets

# ==========================================
# PREPARAÇÃO DO AMBIENTE
# ==========================================
tamanho_msg = 32 # Tamanho igual para ambas as mensagens

# "Crie duas mensagens sintéticas M1 e M2 com o mesmo comprimento"
M1 = secrets.token_bytes(tamanho_msg)
M2 = secrets.token_bytes(tamanho_msg)

# "Gere uma única chave K e, deliberadamente, reutilize-a"
K = gerar_chave(tamanho_msg)

# Cifrando com a MESMA chave
C1 = cifrar(M1, K)
C2 = cifrar(M2, K)

print("--- Parte C: Falha Crítica (Reuso de Chave) ---\n")

# ==========================================
# PASSO 1: Calcule C1 ⊕ C2 e confirme que é igual a M1 ⊕ M2
# ==========================================
# A propriedade mágica do XOR: se você fizer XOR de dois textos cifrados com a mesma chave, 
# a chave se anula e o resultado é o XOR das mensagens originais!
xor_cifrados = xor_bytes(C1, C2)
xor_originais = xor_bytes(M1, M2)

if xor_cifrados == xor_originais:
    print("PASSO 1 [SUCESSO]: Confirmado! C1 ⊕ C2 é exatamente igual a M1 ⊕ M2.")

# ==========================================
# PASSO 2: Assuma que o atacante conhece M1 e C1 e recupere a chave
# ==========================================
# Se o hacker sabe o que estava escrito em M1 (por exemplo, um cabeçalho padrão de rede) 
# e tem o texto cifrado C1, ele acha a chave na hora.
K_recuperada_pelo_hacker = xor_bytes(M1, C1)

if K_recuperada_pelo_hacker == K:
    print("PASSO 2 [SUCESSO]: O atacante recuperou a chave original K perfeitamente!")

# ==========================================
# PASSO 3: Use a chave recuperada para obter M2 a partir de C2
# ==========================================
# Agora que o hacker tem a chave, ele lê a segunda mensagem que deveria ser secreta.
M2_recuperada_pelo_hacker = xor_bytes(C2, K_recuperada_pelo_hacker)

if M2_recuperada_pelo_hacker == M2:
    print("PASSO 3 [SUCESSO]: O atacante decifrou M2 com sucesso usando a chave roubada.")

# ==========================================
# PASSO 5: Repita o teste com chaves independentes
# ==========================================
print("\n--- Teste com Chaves Independentes (O jeito certo) ---")
K1_independente = gerar_chave(tamanho_msg)
K2_independente = gerar_chave(tamanho_msg)

# Cifrando corretamente, cada mensagem com sua própria chave
C1_seguro = cifrar(M1, K1_independente)
C2_seguro = cifrar(M2, K2_independente)

# O atacante tenta o mesmo truque do Passo 2
chave_falsa_recuperada = xor_bytes(M1, C1_seguro) # Ele recuperou K1

# O atacante tenta ler M2 usando K1 (Passo 3)
tentativa_de_ler_M2 = xor_bytes(C2_seguro, chave_falsa_recuperada)

if tentativa_de_ler_M2 != M2:
    print("PASSO 5 [SUCESSO]: A invasão falhou! Como as chaves eram independentes, o hacker obteve apenas lixo numérico ao tentar decifrar M2.")

--- Parte C: Falha Crítica (Reuso de Chave) ---

PASSO 1 [SUCESSO]: Confirmado! C1 ⊕ C2 é exatamente igual a M1 ⊕ M2.
PASSO 2 [SUCESSO]: O atacante recuperou a chave original K perfeitamente!
PASSO 3 [SUCESSO]: O atacante decifrou M2 com sucesso usando a chave roubada.

--- Teste com Chaves Independentes (O jeito certo) ---
PASSO 5 [SUCESSO]: A invasão falhou! Como as chaves eram independentes, o hacker obteve apenas lixo numérico ao tentar decifrar M2.


O experimento viola a regra de ouro do algoritmo One-Time Pad (OTP): a chave deve ser usada única e exclusivamente para uma única mensagem e depois descartada. A segurança perfeita do OTP depende da aleatoriedade absoluta da chave para ocultar a mensagem. No entanto, quando reutilizamos a mesma chave para cifrar duas mensagens diferentes ($C_1$ e $C_2$), criamos uma vulnerabilidade matemática gravíssima. Como a operação XOR é reversível e cancela valores duplicados, ao realizarmos o XOR entre os dois textos cifrados ($C_1 \oplus C_2$), a chave idêntica se anula, deixando como resultado o XOR das duas mensagens em texto claro ($M_1 \oplus M_2$). A partir desse momento, a proteção da chave deixa de existir. Se um atacante descobrir ou deduzir o conteúdo de apenas uma das mensagens, ele pode aplicar a operação XOR para extrair a chave original e, consequentemente, quebrar o sigilo de todas as outras mensagens que compartilharam essa mesma chave.